# Imagens da manhã

Painel de mercado e calendário econômico, e nada além disso. É a fachada de notebook
do `uv run matinal imagens`, para quem vai escrever o texto por fora. Era o que o
repositório `daily` servia, com outro código e as mesmas duas imagens; ele foi
aposentado quando este caminho nasceu.

**Quem vai rodar o plantão inteiro usa o `plantao.ipynb`.** Lá o Passo 1 continua
gerando também o bloco direcional, que é insumo das três etapas de IA.

**Roda a qualquer hora.** A janela de 7h00 às 9h00 é do comentário matinal, e este
notebook não o produz: não há aqui faixa de ensaio nem aviso de dry run. As imagens
também não saem carimbadas — o carimbo vive no bloco direcional, que este notebook
não gera.

Exige o terminal Bloomberg **aberto e logado**, e Windows nativo, nunca WSL: o
`blpapi` conversa com o terminal por IPC local.

In [ ]:
%matplotlib inline

from comentario_matinal import plantao


def mostra(mensagem):
    """O `progresso` das células: cada aviso aparece enquanto o passo roda.

    Não há aqui o dicionário de remédios do `plantao.ipynb`: aquelas frases
    ensinam a suprir etapa que faltou, e neste notebook não há etapa alguma —
    a coleta que os passos seguintes consomem é feita na célula abaixo.
    """
    print(mensagem, flush=True)


ctx = plantao.contexto()

# Sem a faixa de dry run do `plantao.ipynb`, e sem o aviso de dry run do núcleo: a
# janela de 7h00 às 9h00 é do comentário que vai à diretoria, e estas imagens não
# são ele. Rodar às 15h é uso normal daqui, e "não enviar o resultado" seria
# conselho sobre outra coisa. Os demais avisos do contexto — divergência de fuso,
# por exemplo — continuam saindo, porque esses valem para qualquer execução.
for aviso in ctx.avisos:
    if getattr(aviso, "codigo", None) != plantao.DRY_RUN:
        mostra(aviso)

print(f"As imagens vão para {ctx.saida}")

## Coleta

É o único ponto que toca a Bloomberg. Uma consulta só, que serve às duas imagens:
elas descrevem os mesmos números, por construção.

O horário desta execução é carimbado no painel. Se as imagens forem geradas bem
antes de o texto ser escrito, vale regerá-las — direção de ativo muda.

In [ ]:
mercado = plantao.coleta_mercado(ctx, progresso=mostra)

print(f"{len(mercado.referencia)} ativo(s) com referência; "
      f"{len(mercado.indisponiveis)} sem.")
mercado.referencia.head()

## Painel

A grade de ativos, como vai para o e-mail. Ativo sem negociação na sessão sai com o
selo de mercado fechado: a direção mostrada é a da sessão anterior, e não pode ser
descrita como movimento de hoje.

In [ ]:
painel = plantao.desenha_painel(ctx, mercado)

print(f"Painel:     {painel.caminho}")
painel.figura

## Calendário

A tabela de divulgações do dia e de falas de bancos centrais, apurada contra o
horário desta execução. Dado ainda não divulgado aparece como tal — e é assim que
ele não vira fato consumado no texto que você vai escrever.

In [ ]:
# Numa máquina sem licença BQL a consulta do calendário estoura. Trocar para True
# pula o passo, como o `--sem-calendario` do terminal: sai só o painel.
SEM_CALENDARIO = False

calend = None
if not SEM_CALENDARIO:
    calend = plantao.prepara_calendario(ctx, progresso=mostra)

    # Os avisos do calendário nascem depois da consulta, e não passam por `progresso`.
    for aviso in calend.avisos:
        mostra(aviso)

    if calend.caminho_png:
        print(f"Calendário: {calend.caminho_png}")
    print(f"Calendário: {calend.caminho_md}")
else:
    print("Calendário pulado (SEM_CALENDARIO). Sai só o painel.")

calend.eco if calend is not None else None

## Pronto

As duas imagens estão em `output/comentario_matinal/`, com a data do dia no nome — `painel_AAAAMMDD.png`
e `calendario_AAAAMMDD.png`. São essas que se colam no e-mail.

O que **não** foi gerado é o bloco direcional, o `painel_AAAAMMDD.txt`: ele existe
para as etapas de IA lerem o estado do mercado, e quem escreve o texto por fora não
tem o que fazer com ele. Para o plantão inteiro — triagem, redação, revisão e
montagem do documento —, o notebook é o `plantao.ipynb`.